# 00 - data audit, `elections`

Coverage, missingness, dtypes, units, precinct-size distribution, region counts and the
per-row arithmetic identity, for the two precinct files this project rests on.

**There is no inference in this notebook and there must never be any.** Nothing here fits a
model, tests a hypothesis or estimates an effect. Its whole job is to establish that the data
loaded is the data the documentation describes, so that the tests in
`src/elections/analysis/` can be run against something known. The replication targets are in
`docs/validation_anchors.md`; the confounds that make a naive result meaningless are in
`docs/known_traps.md`.

If `data/raw` is empty, every cell below prints what to run and does nothing else.

In [ ]:
from __future__ import annotations

import pandas as pd
from elections.clean import (
    CANONICAL_COLUMNS,
    MISSING_CANONICAL_COLUMNS,
    IntegrityError,
    RawFileMissing,
    add_derived,
    build_tidy,
    run_all_checks,
)
from elections.clean.schema import ELECTION_DATES, EXPECTED_N_COLUMNS
from elections.features import (
    DEFAULT_MIN_DENOMINATOR,
    denominator_floor_summary,
    exact_integer_share,
    percentage_resolution,
    size_band_counts,
)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)

RUN_FIRST = (
    "data/raw is empty. From projects/elections run:\n"
    "    make data     # acquire the sources listed in data/SOURCES.yaml\n"
    "and then re-run this notebook. Nothing is computed until then."
)

# validate=False on purpose: this notebook audits the data, including data that fails a
# check. The checks are run explicitly further down and their failure is reported, not raised.
try:
    RAW = build_tidy(validate=False)
    TIDY = add_derived(RAW)
except RawFileMissing as exc:
    RAW = TIDY = None
    print(RUN_FIRST)
    print()
    print(exc)

## Coverage: rows, elections, regions, commissions

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    coverage = (
        TIDY.groupby("election")
        .agg(
            stations=("uik", "size"),
            regions=("region", "nunique"),
            commissions=("tik", "nunique"),
            registered_total=("registered", "sum"),
            winner_votes_total=("winner_votes", "sum"),
        )
        .assign(polling_day=lambda t: [ELECTION_DATES[e] for e in t.index])
        .assign(raw_columns=lambda t: [EXPECTED_N_COLUMNS[e] for e in t.index])
    )
    display(coverage)

## Which regions appear in one election but not the other

The 2018 file adds Crimea and Sevastopol and splits out Baikonur, so any 2011-to-2018 panel
has to decide what to do with regions that have no counterpart. This cell only names them.

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    by_election = {e: set(g["region"].dropna()) for e, g in TIDY.groupby("election")}
    elections = sorted(by_election)
    if len(elections) == 2:
        left, right = elections
        print(f"in {left} only : {sorted(by_election[left] - by_election[right])}")
        print(f"in {right} only: {sorted(by_election[right] - by_election[left])}")
        print(f"in both        : {len(by_election[left] & by_election[right])} regions")
    stations_per_region = (
        TIDY.groupby(["election", "region"]).size().rename("stations").reset_index()
    )
    display(stations_per_region.sort_values("stations", ascending=False).head(15))
    display(stations_per_region.sort_values("stations").head(15))

## Missingness

A column that an election never had is expected to be entirely null: those columns are named
in `MISSING_CANONICAL_COLUMNS` and are listed separately from anything unexpected. Only the
second table is a problem.

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    rows = []
    for election, group in RAW.groupby("election"):
        expected_null = set(MISSING_CANONICAL_COLUMNS[election])
        for column in CANONICAL_COLUMNS:
            n_null = int(group[column].isna().sum())
            rows.append(
                {
                    "election": election,
                    "column": column,
                    "n_null": n_null,
                    "share_null": n_null / len(group) if len(group) else float("nan"),
                    "expected": column in expected_null,
                }
            )
    missingness = pd.DataFrame(rows)
    print("columns absent by design (should be 100% null):")
    display(missingness.loc[missingness["expected"] & (missingness["n_null"] > 0)])
    print("unexpected missingness (should be empty):")
    display(missingness.loc[~missingness["expected"] & (missingness["n_null"] > 0)])

## Dtypes and units

Every count is a nullable integer in units of *ballots* or *voters*, except `uik`, which is an
identifier that happens to be numeric and must never be summed. The derived columns are
fractions in [0, 1], not percentages; `forensics_core.digits.integer_pct` wants the 0-100
scale, so it has to be asked for explicitly.

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    units = {
        "election": "election key",
        "region": "federal subject name",
        "tik": "territorial commission name",
        "uik": "station number (identifier, do not sum)",
        "winner_label": "contestant of record",
        "source_url": "commission portal URL the row was scraped from",
        "turnout_boxes": "fraction of registered voters",
        "turnout_issued": "fraction of registered voters",
        "winner_share": "fraction of ballots counted",
    }
    dtypes = pd.DataFrame(
        {
            "dtype": TIDY.dtypes.astype(str),
            "unit": [units.get(c, "count of ballots or voters") for c in TIDY.columns],
        }
    )
    display(dtypes)
    print("derived scale:", TIDY.attrs.get("derived_scale"))

## Precinct size, and how much of the sample a denominator floor removes

Trap 1 in `docs/known_traps.md`. `exact_integer_share` is arithmetic on the station size
alone: the fraction of percentages a station of that size can even attain that are whole
numbers. It is what makes the floor necessary rather than fussy.

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    display(
        TIDY.groupby("election")["registered"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
    )
    display(size_band_counts(TIDY))
    for election, group in TIDY.groupby("election"):
        summary = denominator_floor_summary(
            group["registered"], min_denominator=DEFAULT_MIN_DENOMINATOR
        )
        print(
            f"{election}: floor {summary.min_denominator} removes {summary.n_below:,} of "
            f"{summary.n_total:,} stations ({summary.share_below:.2%}); "
            f"{summary.n_missing:,} have no recorded size"
        )

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    sizes = TIDY["registered"]
    grid = pd.DataFrame(
        {
            "registered": sizes,
            "percentage_resolution": percentage_resolution(sizes).to_numpy(),
            "exact_integer_share": exact_integer_share(sizes).to_numpy(),
        }
    )
    display(grid.describe().T)
    print("stations where every attainable percentage is a whole number:")
    print(int((grid["exact_integer_share"] == 1.0).sum()))

## The arithmetic identity, and the rest of the acceptance checks

`invalid + valid == in_mobile_boxes + in_stationary_boxes` held for all 95,225 rows of the
2011 file when the registry was written. Here the checks are *reported* rather than enforced,
because a notebook that raises tells you less than one that shows you which rows broke.

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    counted = RAW["invalid"] + RAW["valid"]
    found = RAW["in_mobile_boxes"] + RAW["in_stationary_boxes"]
    breaks = (counted != found).fillna(True)
    print(f"rows violating the ballot identity: {int(breaks.sum()):,} of {len(RAW):,}")
    if int(breaks.sum()):
        display(
            RAW.loc[
                breaks,
                [
                    "election",
                    "region",
                    "tik",
                    "uik",
                    "invalid",
                    "valid",
                    "in_mobile_boxes",
                    "in_stationary_boxes",
                ],
            ].head(20)
        )

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    for election, group in RAW.groupby("election"):
        try:
            print(run_all_checks(group, election))
        except IntegrityError as exc:
            print(f"{election}: FAILED -- {exc}")

## Impossible values

Not a test of anything: a count of rows whose arithmetic is outside what the protocol allows.
Trap 4 says scraped hierarchical tables produce these, and that they must be logged before any
analysis rather than dropped silently inside one.

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    flags = pd.DataFrame(
        {
            "election": TIDY["election"],
            "turnout_boxes_above_1": TIDY["turnout_boxes"] > 1,
            "turnout_issued_above_1": TIDY["turnout_issued"] > 1,
            "winner_share_above_1": TIDY["winner_share"] > 1,
            "no_registered_voters": TIDY["registered"] <= 0,
            "counted_exceeds_issued": (TIDY["invalid"] + TIDY["valid"])
            > (TIDY["ballots_early"] + TIDY["ballots_in_station"] + TIDY["ballots_outside"]),
        }
    )
    display(flags.groupby("election").sum(numeric_only=False))

## The raw headers, as read

Kept in the frame's `attrs` by the loader. The six 2011 absentee-certificate columns are named
`abs_line_1` .. `abs_line_6` because their individual labels were never transcribed into the
registry; this is where to read what they actually say before using any one of them alone.

In [ ]:
if TIDY is None:
    print(RUN_FIRST)
else:
    for election, header in RAW.attrs.get("raw_headers", {}).items():
        print(f"--- {election}: {len(header)} columns")
        for index, label in enumerate(header):
            print(f"  [{index:2d}] {label}")